In [ ]:
# [A] Mount Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"   # your project base folder

# ---- INPUTS ----
FORUM_CSV   = f"{BASE}/raw/forums/multi_forum_property_posts_hwz.csv"

# Singlish lexicon (rich version)
SINGLEX_CSV = f"{BASE}/corpus/Singlish/lexicon.csv"

# Property domain resources (SGPropertyDomain)
ENTITYRULER = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"

# ---- OUTPUTS ----
OUTDIR = f"{BASE}/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025"


Mounted at /content/drive


In [ ]:
!pip -q install pandas pyarrow spacy


In [ ]:
from pathlib import Path
import pandas as pd
import os, re, json, html, unicodedata


In [ ]:
# === Auto-merge SGPropertyDomain vocab/*.txt into EntityRuler patterns ===
# PLACE: after your PATHS are defined, before helpers/main run


# If your paths differ, adjust BASE / VOC_DIR / EXISTING.
BASE     = "/content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain"
VOC_DIR  = f"{BASE}/vocab"                                  # contains *.txt like HDB.txt, Finance&Rates.txt, ...
EXISTING = f"{BASE}/spacy_entityruler_patterns.jsonl"       # your current file (ok if missing)
MERGED   = f"{BASE}/spacy_entityruler_patterns.merged.jsonl" # output we will generate

def iter_existing(path):
    out = []
    p = Path(path)
    if not p.exists():
        return out
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except:
                pass
    return out

def phrase_to_token_pattern(phrase: str):
    """
    Case-insensitive token pattern for spaCy EntityRuler.
    Supports multi-word phrases and keeps hyphens/apostrophes inside tokens.
    """
    phrase = re.sub(r"\s+", " ", phrase).strip()
    if not phrase:
        return None
    tokens = phrase.split(" ")
    return [{"LOWER": t.lower()} for t in tokens if t]

def label_from_filename(fname: str):
    # e.g., "Finance&Rates.txt" -> "FINANCE_RATES"
    stem  = Path(fname).stem
    label = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").upper()
    return label or "DOMAIN"

# 1) Existing patterns (keep priority)
existing = iter_existing(EXISTING)
print(f"[INFO] Loaded existing patterns: {len(existing)}")

# 2) Build from vocab/*.txt
voc_path = Path(VOC_DIR)
assert voc_path.exists(), f"Vocab directory not found: {VOC_DIR}"

vocab_patterns = []
for txt in sorted(voc_path.glob("*.txt")):
    label = label_from_filename(txt.name)
    for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
        term = raw.strip()
        if not term:
            continue
        pat = phrase_to_token_pattern(term)
        if not pat:
            continue
        vocab_patterns.append({"label": label, "pattern": pat, "id": term})

print(f"[INFO] Built vocab patterns: {len(vocab_patterns)}")

def lowers_from_pattern(pat):
    """
    Normalize spaCy EntityRuler 'pattern' into a tuple of lowercase tokens for de-dup keys.
    Supports:
      - str: "option to purchase"
      - dict: {"LOWER": "absd"} or {"TEXT": "ABSD"} (treated as 1 token)
      - list of dicts: [{"LOWER":"option"}, {"LOWER":"to"}, {"LOWER":"purchase"}]
    """
    if isinstance(pat, str):
        return tuple(p.strip().lower() for p in re.sub(r"\s+", " ", pat).split(" ") if p.strip())
    if isinstance(pat, dict):
        return (str(pat.get("LOWER", pat.get("TEXT", ""))).lower(),)
    if isinstance(pat, list):
        outs = []
        for tok in pat:
            if isinstance(tok, dict):
                outs.append(str(tok.get("LOWER", tok.get("TEXT", ""))).lower())
            else:
                outs.append(str(tok).lower())
        return tuple(outs)
    # fallback
    return (str(pat).lower(),)

def pat_key(rec):
    pat = rec.get("pattern", "")
    label = rec.get("label", "")
    return (label, lowers_from_pattern(pat))

seen, merged = set(), []
str_count = 0
list_count = 0
dict_count = 0

# add existing first (so they have priority)
for rec in existing:
    # quick stats
    if isinstance(rec.get("pattern", ""), str): str_count += 1
    elif isinstance(rec.get("pattern", ""), list): list_count += 1
    elif isinstance(rec.get("pattern", ""), dict): dict_count += 1
    k = pat_key(rec)
    if k in seen:
        continue
    seen.add(k)
    merged.append(rec)

# then add vocab-derived ones if not already present
skipped = 0
for rec in vocab_patterns:
    k = pat_key(rec)
    if k in seen:
        skipped += 1
        continue
    seen.add(k)
    merged.append(rec)

print(f"[INFO] Existing pattern types — str:{str_count}, list:{list_count}, dict:{dict_count}")
print(f"[INFO] Merged total patterns: {len(merged)} (skipped dupes: {skipped})")

# 4) Save JSONL (keep original formats; spaCy supports string or token-pattern forms)
with open(MERGED, "w", encoding="utf-8") as f:
    for rec in merged:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"[DONE] Wrote merged patterns → {MERGED}")

# 5) IMPORTANT: point your pipeline to the NEW merged file
ENTITYRULER = MERGED
print("ENTITYRULER now set to:", ENTITYRULER)
print(f"[INFO] Merged total patterns: {len(merged)} (skipped dupes: {skipped})")



[INFO] Loaded existing patterns: 2256
[INFO] Built vocab patterns: 1025
[INFO] Existing pattern types — str:2256, list:0, dict:0
[INFO] Merged total patterns: 2158 (skipped dupes: 795)
[DONE] Wrote merged patterns → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
ENTITYRULER now set to: /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
[INFO] Merged total patterns: 2158 (skipped dupes: 795)


#Helpers (cleaning + loaders)

In [ ]:

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def basic_clean(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"(https?://\S+|www\.\S+)", " ", s)
    s = re.sub(r"[\[\]{}<>]", " ", s)
    return normalize_ws(s)

def load_jsonl(path: Path):
    items=[]
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if ln:
                try: items.append(json.loads(ln))
                except: pass
    return items

def compile_regexes_from_jsonl(path: Path):
    patt=[]
    for it in load_jsonl(path):
        pat = it.get("pattern")
        if not pat: continue
        try: patt.append({"name": it.get("name","pattern"), "re": re.compile(pat)})
        except re.error: pass
    return patt

def detect_regex_hits(text: str, compiled):
    hits={}
    for p in compiled:
        try:
            if p["re"].search(text): hits[p["name"]] = True
        except: pass
    return hits

# ---- Singlish loaders ----

def build_singdict_from_lexicon(csv_path: str):
    """
    Reads your lexicon.csv (columns like 'Word', 'Description', etc.)
    Returns:
      words_set: set of lowercase terms for fast matching
      meta_map : dict term -> {description: "..."} (for optional meanings)
    """
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column not found in {csv_path}. Columns: {df.columns.tolist()}")
    df = df.dropna(subset=["word"])
    words_set = set()
    meta_map  = {}
    for _, r in df.iterrows():
        w = str(r["word"]).strip().lower()
        if not w: continue
        words_set.add(w)
        desc = str(r.get("description", "")).strip()
        meta_map[w] = {"description": desc} if desc else {}
    return words_set, meta_map

def find_singlish_terms(text: str, words_set):
    if not words_set: return [], text
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+|\d+|[^\w\s]", text)
    found, out = [], []
    for tok in toks:
        low = tok.lower()
        if low in words_set:
            found.append(low); out.append(low)   # keep as-is; you can map to a normalized form if desired
        else:
            out.append(tok)
    return sorted(list(set(found))), normalize_ws(" ".join(out))

def add_entity_ruler_spacy(df: pd.DataFrame, text_col: str, patterns_path: str):
    try:
        import spacy
        nlp = spacy.blank("en")
        ruler = nlp.add_pipe("entity_ruler")
        ruler.from_disk(str(patterns_path))
        ents=[]
        for doc in nlp.pipe(df[text_col].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"] = ents
    except Exception as e:
        print(f"[WARN] spaCy entity_ruler skipped: {e}")
        df["entities"] = [[] for _ in range(len(df))]
    return df


#Main preprocessing (clean → dedupe → Singlish → Property)

In [ ]:

assert Path(FORUM_CSV).exists(), f"Forum file missing: {FORUM_CSV}"
assert Path(SINGLEX_CSV).exists(), f"Lexicon file missing: {SINGLEX_CSV}"


In [ ]:

# ---- load forum data
df = pd.read_csv(FORUM_CSV)

print(df.columns.tolist())
print(df.head(3).to_dict(orient="records"))
print(df["thread_url"].head(10).tolist() if "thread_url" in df.columns else "No thread_url col")


['forum', 'thread_url', 'post_text']
[{'forum': 'HardwareZone', 'thread_url': 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'post_text': 'Not sure if this been posted or not ? hdb resales transactions private ppty transactions'}, {'forum': 'HardwareZone', 'thread_url': 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'post_text': "i have posted few times on the hdb resale transaction's link bt not the pte ppty one. will put it as sticky. =p"}, {'forum': 'HardwareZone', 'thread_url': 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'post_text': 'CPF CPF Home ownership Handbook CPF Home Ownership Scheme HDB Renovation/Housing Maintenance Flat Ownership Info Resales Of flats Rental of Flats'}]
['https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'https://forums.hardwar

In [ ]:

# ----------------------------
# 1) LOAD + NORMALIZE COLUMNS
# ----------------------------

# text column -> post_text
TEXT_COL_CANDS = ["post_text", "text", "content", "body"]
TEXT_COL = next((c for c in TEXT_COL_CANDS if c in df.columns), None)
assert TEXT_COL is not None, f"No text column found. Columns: {list(df.columns)}"
if TEXT_COL != "post_text":
    df = df.rename(columns={TEXT_COL: "post_text"})

# forum -> forum_name (optional)
if "forum_name" not in df.columns and "forum" in df.columns:
    df = df.rename(columns={"forum": "forum_name"})

# ------------- OPTIONAL DATE -------------
# Try to build a 'date' column if possible (but do not filter by year)
DATE_COL_CANDS = ["date", "posted_at", "post_date", "created_at", "timestamp", "time"]
DATE_COL = next((c for c in DATE_COL_CANDS if c in df.columns), None)

def try_parse_dt(x):
    return pd.to_datetime(x, errors="coerce")

def extract_date_from_url_like(s: str):
    """Try multiple URL patterns incl. UNIX timestamps; else NaT."""
    if not isinstance(s, str):
        return pd.NaT
    # yyyy-mm-dd
    m = re.search(r"(20\d{2})-(0?[1-9]|1[0-2])-(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    # yyyy/mm/dd
    m = re.search(r"(20\d{2})/(0?[1-9]|1[0-2])/(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    # yyyy_mm_dd
    m = re.search(r"(20\d{2})_(0?[1-9]|1[0-2])_(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    # yyyymmdd
    m = re.search(r"\b(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])\b", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")
    # UNIX epoch seconds (10) or ms (13)
    m = re.search(r"(?<!\d)(1[5-9]\d{8}|2\d{9})(?!\d)", s)
    if m:
        return pd.to_datetime(int(m.group(1)), unit="s", errors="coerce", utc=True).tz_localize(None)
    m = re.search(r"(?<!\d)(1[5-9]\d{11}|2\d{12})(?!\d)", s)
    if m:
        return pd.to_datetime(int(m.group(1)), unit="ms", errors="coerce", utc=True).tz_localize(None)
    return pd.NaT

if DATE_COL is not None:
    df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
else:
    # try to derive from URL-like columns, but do NOT enforce
    url_cols = [c for c in ["thread_url", "post_url", "url", "source_url"] if c in df.columns]
    if url_cols:
        dates = pd.Series(pd.NaT, index=df.index)
        for c in url_cols:
            cand = df[c].astype(str).apply(extract_date_from_url_like)
            dates = dates.fillna(cand)
        df["date"] = dates
    else:
        df["date"] = pd.NaT

HAS_DATE = df["date"].notna().any() if "date" in df.columns else False
print(f"[INFO] HAS_DATE = {HAS_DATE}  (non-null dates: {df['date'].notna().sum() if 'date' in df.columns else 0})")

# -------------------------
# 2) CLEAN + DEDUP (no year filter)
# -------------------------
df["raw_text"]   = df["post_text"].astype(str)
df["clean_text"] = df["raw_text"].apply(basic_clean)
df = df[df["clean_text"].str.len() > 20].copy()

# dedupe: title? + author? + clean_text (+ date(day) if available)
key_parts = [df["clean_text"].astype(str)]
if "author" in df.columns:
    key_parts.insert(0, df["author"].astype(str))
if "title" in df.columns:
    key_parts.insert(0, df["title"].astype(str))
if HAS_DATE:
    key_parts.append(df["date"].dt.date.astype(str))

df["_k"] = key_parts[0]
for col in key_parts[1:]:
    df["_k"] = df["_k"] + "||" + col

before = len(df)
df = df.drop_duplicates(subset=["_k"]).drop(columns=["_k"])
after = len(df)
print(f"[INFO] Dedupe removed {before - after} rows  →  kept {after}")

# ---------------------------------
# 3) SINGLISH + PROPERTY ENRICHMENT
# ---------------------------------
# Singlish (from lexicon.csv)
sing_words, sing_meta = build_singdict_from_lexicon(SINGLEX_CSV)

found_terms, norm_texts, meanings = [], [], []
for t in df["raw_text"].astype(str):
    f, n = find_singlish_terms(t, sing_words)
    found_terms.append(f)
    norm_texts.append(n)
    m = [sing_meta[w]["description"] for w in f if w in sing_meta and "description" in sing_meta[w] and sing_meta[w]["description"]]
    meanings.append(list(dict.fromkeys(m)))
df["singlish_terms"]    = found_terms
df["has_singlish"]      = df["singlish_terms"].apply(bool)
df["text_sing_norm"]    = norm_texts
df["singlish_meanings"] = meanings

# Property regex flags
if REGEX_JSONL and Path(REGEX_JSONL).exists():
    comp = compile_regexes_from_jsonl(Path(REGEX_JSONL))
    hits = [detect_regex_hits(t, comp) for t in df["clean_text"].astype(str)]
    hdf  = pd.json_normalize(hits)
    hdf.columns = [f"rx_{c}" for c in hdf.columns]
    df = pd.concat([df.reset_index(drop=True), hdf.reset_index(drop=True)], axis=1)

# EntityRuler domain entities
if ENTITYRULER and Path(ENTITYRULER).exists():
    df = add_entity_ruler_spacy(df, "clean_text", ENTITYRULER)
else:
    df["entities"] = [[] for _ in range(len(df))]

# quick stats
df["num_chars"] = df["clean_text"].str.len()
df["num_words"] = df["clean_text"].str.split().apply(len)
print(f"[STATS] total rows: {len(df)} | posts with Singlish: {df['has_singlish'].sum()}")

# ---------------
# 4) SAVE CORE
# ---------------
Path(OUTDIR).mkdir(parents=True, exist_ok=True)
out_csv  = f"{OUTDIR}/forum_enriched.csv"          # generic (no 2023_2025)
out_parq = f"{OUTDIR}/forum_enriched.parquet"
df.to_csv(out_csv, index=False)
try:
    df.to_parquet(out_parq, index=False)
except Exception as e:
    print(f"[WARN] Parquet write failed: {e}")

stats = {
    "rows_after_clean_dedup": int(len(df)),
    "has_singlish_true": int(df["has_singlish"].sum()),
    "cols": list(df.columns),
    "has_date": bool(HAS_DATE),
    "non_null_dates": int(df["date"].notna().sum()) if "date" in df.columns else 0,
}
with open(f"{OUTDIR}/preprocess_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print("Saved:", out_csv)
print("Saved:", out_parq)
print("Saved:", f"{OUTDIR}/preprocess_stats.json")



[INFO] HAS_DATE = False  (non-null dates: 0)
[INFO] Dedupe removed 10 rows  →  kept 27530
[STATS] total rows: 27530 | posts with Singlish: 21978
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched.csv
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/preprocess_stats.json


In [ ]:
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 144.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# === Sentence segmentation + general NER + ABSA scaffolding + temporal trends ===

# ---------------------------------------------------
# 5) NLP ENRICHMENT (tokens/lemmas/NER/ABSA)
# ---------------------------------------------------

# 5.1 spaCy pipeline

import spacy
nlp = spacy.load("en_core_web_sm", exclude=[])  # tagger, parser, NER
# re-attach domain EntityRuler BEFORE 'ner'
try:
    if ENTITYRULER and Path(ENTITYRULER).exists():
        ruler = nlp.add_pipe("entity_ruler", before="ner")
        ruler.from_disk(ENTITYRULER)
except Exception as e:
    print("[WARN] EntityRuler reattach skipped:", e)

# 5.2 docs
docs = list(nlp.pipe(df["clean_text"].astype(str).tolist(), batch_size=64, n_process=2))

# 5.3 token/POS/Dep
df["tokens"] = [[t.text  for t in d] for d in docs]
df["lemmas"] = [[t.lemma_ for t in d] for d in docs]
df["pos"]    = [[t.pos_   for t in d] for d in docs]
df["deps"]   = [[t.dep_   for t in d] for d in docs]

# 5.4 sentence table (include date only if we have it)
sent_rows = []
use_date = HAS_DATE
for i, d in enumerate(docs):
    for j, s in enumerate(d.sents):
        row = {
            "post_id": i,
            "sent_id": j,
            "text": s.text,
            "tokens": [t.text for t in s],
            "lemmas": [t.lemma_ for t in s],
            "pos":    [t.pos_   for t in s],
            "deps":   [t.dep_   for t in s],
        }
        if use_date:
            row["date"] = df.iloc[i]["date"]
        sent_rows.append(row)

sent_df = pd.DataFrame(sent_rows)
sent_path = f"{OUTDIR}/forum_sentences.parquet"
sent_df.to_parquet(sent_path, index=False)
print("Saved sentence table →", sent_path)

# 5.5 general NER + aspects (np chunks)
df["entities_ner"] = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
df["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]
df.to_parquet(f"{OUTDIR}/forum_enriched+nlp.parquet", index=False)
print("Saved:", f"{OUTDIR}/forum_enriched+nlp.parquet")

# 5.6 monthly trend only if dates exist
if HAS_DATE:
    monthly = (
        df.assign(month=df["date"].dt.to_period("M").astype(str))
          .groupby("month", dropna=True)
          .agg(posts=("clean_text", "size"),
               with_singlish=("has_singlish", "sum"))
          .reset_index()
    )
    monthly.to_csv(f"{OUTDIR}/monthly_counts.csv", index=False)
    print("Saved:", f"{OUTDIR}/monthly_counts.csv")
else:
    print("[INFO] Skipping monthly trend: no usable dates.")

# 5.7 quick file check
print("OUTDIR:", OUTDIR)
print("Files:", sorted(os.listdir(OUTDIR))[:12])



Saved sentence table → /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_sentences.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched+nlp.parquet
[INFO] Skipping monthly trend: no usable dates.
OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025
Files: ['forum_enriched+nlp.parquet', 'forum_enriched.csv', 'forum_enriched.parquet', 'forum_sentences.parquet', 'preprocess_stats.json']


#Verification

In [ ]:


print("OUTDIR:", OUTDIR)
print("Files:", sorted(os.listdir(OUTDIR))[:12])

# --- check the main enriched file
enriched_csv = f"{OUTDIR}/forum_enriched.csv"
if os.path.exists(enriched_csv):
    df_check = pd.read_csv(enriched_csv, nrows=3)
    print("\n✅ Loaded:", enriched_csv)
    print("Columns:", df_check.columns.tolist())
else:
    print("\n❌ forum_enriched.csv not found!")

# --- check monthly trends (only present if HAS_DATE == True)
monthly_path = f"{OUTDIR}/monthly_counts.csv"
print("Has monthly_counts.csv?", os.path.exists(monthly_path))

# --- check sentence-level parquet
sent_path = f"{OUTDIR}/forum_sentences.parquet"
print("Has sentence parquet?", os.path.exists(sent_path))

# --- check enriched+nlp parquet
nlp_path = f"{OUTDIR}/forum_enriched+nlp.parquet"
print("Has enriched+nlp parquet?", os.path.exists(nlp_path))

# --- check stats JSON
stats_path = f"{OUTDIR}/preprocess_stats.json"
if os.path.exists(stats_path):
    print("\nStats JSON contents:")
    print(json.load(open(stats_path)))
else:
    print("\nStats file missing.")



OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025
Files: ['forum_enriched+nlp.parquet', 'forum_enriched.csv', 'forum_enriched.parquet', 'forum_sentences.parquet', 'preprocess_stats.json']

✅ Loaded: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched.csv
Columns: ['forum_name', 'thread_url', 'post_text', 'date', 'raw_text', 'clean_text', 'singlish_terms', 'has_singlish', 'text_sing_norm', 'singlish_meanings', 'entities', 'num_chars', 'num_words']
Has monthly_counts.csv? False
Has sentence parquet? True
Has enriched+nlp parquet? True

Stats JSON contents:
{'rows_after_clean_dedup': 27530, 'has_singlish_true': 21978, 'cols': ['forum_name', 'thread_url', 'post_text', 'date', 'raw_text', 'clean_text', 'singlish_terms', 'has_singlish', 'text_sing_norm', 'singlish_meanings', 'entities', 'num_chars', 'num_words'], 'has_date': False, 'non_null_dates': 0}


In [ ]:
FORUM_CSV = f"{BASE}/preprocess/raw/forums/singapore_property_forum_posts_sgexpats.csv"
OUTDIR    = f"{BASE}/preprocess/forum/singapore_property_forum_posts_sgexpats_processed"

In [ ]:
# === SG forum boilerplate cleaner + (optional) embedded date from `post_text` ===
# Place: AFTER `df = pd.read_csv(FORUM_CSV)` and BEFORE your main cleaning/dedupe/enrichment.

import re
import pandas as pd

# 1) Regex cleaner that strips forum noise/boilerplate from `post_text`
NOISE_PATTERNS = [
    # Quote / Reply fragments (remove "Quote..." up to the next "Post by" or end)
    (r"(?is)\bquote\b.*?(?=\bpost by\b|\Z)", " "),
    # Login / Like prompts
    (r"(?i)\blogin to like this post\b", " "),
    (r"(?i)\breport this post\b", " "),
    # Post metadata headers: "Post by <user> » ... am/pm"
    (r"(?is)\bpost by\s+.*?»\s*.*?(?:\bam\b|\bpm\b)", " "),
    # User mentions / ranks / profile metadata
    (r"(?i)\bjoined:\s*.*?\d{4}\b", " "),
    (r"(?i)\bposts:\s*\d+\b", " "),
    (r"(?i)\b(member|moderator|admin|super\s*moderator)\b", " "),
    # Thread navigation text
    (r"(?i)\b(back to top|next post|previous post)\b", " "),
    # Directional arrows / quote symbols
    (r"[»«><]+", " "),
    # Stray "Quote from ..." or "Quote:"
    (r"(?i)\bquote from\b.*?:", " "),
    (r"(?i)\bquote:\b", " "),
    # HTML leftovers (quick strip here; your basic_clean also unescapes HTML)
    (r"(?i)&nbsp;|<br\s*/?>|</?p>", " "),
    # Empty markers / numeric log actions
    (r"(?i)\b0\s+login to like this post\b", " "),
]

def clean_forum_noise(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text
    for pat, rep in NOISE_PATTERNS:
        t = re.sub(pat, rep, t)
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Apply noise cleaner if `post_text` exists
assert "post_text" in df.columns, f"`post_text` column not found. Columns: {list(df.columns)}"
df["post_text"] = df["post_text"].astype(str).apply(clean_forum_noise)

# 2) OPTIONAL: Extract embedded forum date from the cleaned `post_text`
#    Example it will catch: "Tue, 14 Sep 2021 10:00 am"
def extract_forum_date_from_text(text: str):
    if not isinstance(text, str):
        return pd.NaT
    m = re.search(
        r"[A-Z][a-z]{2},\s*\d{1,2}\s+[A-Z][a-z]{2}\s+20\d{2}\s+\d{1,2}:\d{2}\s*(?:am|pm)",
        text
    )
    if not m:
        return pd.NaT
    try:
        return pd.to_datetime(m.group(0), errors="coerce")
    except Exception:
        return pd.NaT

# Only fill `date` from text if you actually want/need it and if `date` is currently all NaT/missing.
if "date" not in df.columns or df["date"].notna().sum() == 0:
    df["date"] = df["post_text"].apply(extract_forum_date_from_text)
    print(f"[INFO] Embedded dates parsed from text: {df['date'].notna().sum()} / {len(df)}")

# 3) Preview a few rows to verify cleaning worked
print("\n--- Sample cleaned rows ---")
for i in range(min(3, len(df))):
    print(f"{i+1}.", df.iloc[i]["post_text"][:200], "\n")


In [ ]:

FORUM_CSV = f"{BASE}/raw/forums/singapore_property_forum_posts_sgexpats.csv"  # SGExpats
# Output folder (per dataset)
OUTDIR = f"{BASE}/preprocess/forum/{Path(FORUM_CSV).stem}_processed"

# Singlish lexicon + SGPropertyDomain resources
SINGLEX_CSV = f"{BASE}/corpus/Singlish/lexicon.csv"
ENTITYRULER  = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL  = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"

# NLP toggles
RUN_NLP_ENRICHMENT = True
YEAR_FILTER = True
YEAR_MIN, YEAR_MAX = 2023, 2025


In [ ]:



# ---------- Small utils ----------
def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def basic_clean(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"(https?://\S+|www\.\S+)", " ", s)
    s = re.sub(r"[\[\]{}<>]", " ", s)
    return normalize_ws(s)

def load_jsonl(path: Path):
    items = []
    if not path.exists(): return items
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                try: items.append(json.loads(ln))
                except: pass
    return items

def compile_regexes_from_jsonl(path: Path):
    patt=[]
    for it in load_jsonl(path):
        pat = it.get("pattern")
        if not pat: continue
        try: patt.append({"name": it.get("name","pattern"), "re": re.compile(pat)})
        except re.error: pass
    return patt

def detect_regex_hits(text: str, compiled):
    hits={}
    for p in compiled:
        try:
            if p["re"].search(text): hits[p["name"]] = True
        except: pass
    return hits

# Singlish
def build_singdict_from_lexicon(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column not found in {csv_path}. Columns: {df.columns.tolist()}")
    df = df.dropna(subset=["word"])
    words_set, meta_map = set(), {}
    for _, r in df.iterrows():
        w = str(r["word"]).strip().lower()
        if not w: continue
        words_set.add(w)
        desc = str(r.get("description", "")).strip()
        meta_map[w] = {"description": desc} if desc else {}
    return words_set, meta_map

def find_singlish_terms(text: str, words_set):
    if not words_set: return [], text
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+|\d+|[^\w\s]", text)
    found, out = [], []
    for tok in toks:
        low = tok.lower()
        if low in words_set:
            found.append(low); out.append(low)
        else:
            out.append(tok)
    return sorted(list(set(found))), normalize_ws(" ".join(out))

# EntityRuler apply
def add_entity_ruler_spacy(df: pd.DataFrame, text_col: str, patterns_path: str):
    try:
        import spacy
        nlp = spacy.blank("en")
        ruler = nlp.add_pipe("entity_ruler")
        ruler.from_disk(str(patterns_path))
        ents=[]
        for doc in nlp.pipe(df[text_col].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"] = ents
    except Exception as e:
        print(f"[WARN] spaCy entity_ruler skipped: {e}")
        df["entities"] = [[] for _ in range(len(df))]
    return df

# ---------- [2] AUTO-MERGE VOCAB → EntityRuler merged ----------
def merge_vocab_to_entityruler():
    base = f"{BASE}/corpus/SGPropertyDomain"
    voc_dir = f"{base}/vocab"
    existing = f"{base}/spacy_entityruler_patterns.jsonl"
    merged  = f"{base}/spacy_entityruler_patterns.merged.jsonl"

    def iter_existing(path):
        out=[]
        p=Path(path)
        if not p.exists(): return out
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line=line.strip()
                if not line: continue
                try: out.append(json.loads(line))
                except: pass
        return out

    def phrase_to_token_pattern(phrase: str):
        phrase = re.sub(r"\s+", " ", phrase).strip()
        if not phrase: return None
        tokens = phrase.split(" ")
        return [{"LOWER": t.lower()} for t in tokens if t]

    def label_from_filename(fname: str):
        stem = Path(fname).stem
        label = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").upper()
        return label or "DOMAIN"

    existing_rules = iter_existing(existing)
    print(f"[INFO] Loaded existing patterns: {len(existing_rules)}")

    voc_path = Path(voc_dir)
    if not voc_path.exists():
        print(f"[INFO] Vocab dir not found ({voc_dir}); using existing patterns only.")
        Path(merged).write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in existing_rules), encoding="utf-8")
        return merged

    vocab_patterns=[]
    for txt in sorted(voc_path.glob("*.txt")):
        label = label_from_filename(txt.name)
        for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
            term = raw.strip()
            if not term: continue
            pat = phrase_to_token_pattern(term)
            if not pat: continue
            vocab_patterns.append({"label": label, "pattern": pat, "id": term})

    print(f"[INFO] Built vocab patterns: {len(vocab_patterns)}")

    def lowers_from_pattern(pat):
        if isinstance(pat, str):
            return tuple(p.strip().lower() for p in re.sub(r"\s+", " ", pat).split(" ") if p.strip())
        if isinstance(pat, dict):
            return (str(pat.get("LOWER", pat.get("TEXT", ""))).lower(),)
        if isinstance(pat, list):
            outs=[]
            for tok in pat:
                if isinstance(tok, dict):
                    outs.append(str(tok.get("LOWER", tok.get("TEXT",""))).lower())
                else:
                    outs.append(str(tok).lower())
            return tuple(outs)
        return (str(pat).lower(),)

    def pat_key(rec):
        return (rec.get("label",""), lowers_from_pattern(rec.get("pattern","")))

    seen=set()
    merged_rules=[]
    for rec in existing_rules:
        k = pat_key(rec)
        if k in seen: continue
        seen.add(k); merged_rules.append(rec)

    skipped=0
    for rec in vocab_patterns:
        k = pat_key(rec)
        if k in seen:
            skipped += 1; continue
        seen.add(k); merged_rules.append(rec)

    with open(merged, "w", encoding="utf-8") as f:
        for rec in merged_rules:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"[DONE] Wrote merged patterns → {merged} (skipped dupes: {skipped})")
    return merged

from pathlib import Path
ENTITYRULER = merge_vocab_to_entityruler() or ENTITYRULER
print("ENTITYRULER now:", ENTITYRULER)

# ---------- [3] LOAD CSV ----------
assert Path(FORUM_CSV).exists(), f"Forum file missing: {FORUM_CSV}"
assert Path(SINGLEX_CSV).exists(), f"Lexicon file missing: {SINGLEX_CSV}"
df = pd.read_csv(FORUM_CSV)
print("[INFO] Loaded:", FORUM_CSV, "| rows:", len(df))
print("Cols:", df.columns.tolist()[:20])

# ---------- [4] CLEAN FORUM BOILERPLATE (+embed date) ----------
NOISE_PATTERNS = [
    (r"(?is)\bquote\b.*?(?=\bpost by\b|\Z)", " "),
    (r"(?i)\blogin to like this post\b", " "),
    (r"(?i)\breport this post\b", " "),
    (r"(?is)\bpost by\s+.*?»\s*.*?(?:\bam\b|\bpm\b)", " "),
    (r"(?i)\bjoined:\s*.*?\d{4}\b", " "),
    (r"(?i)\bposts:\s*\d+\b", " "),
    (r"(?i)\b(member|moderator|admin|super\s*moderator)\b", " "),
    (r"(?i)\b(back to top|next post|previous post)\b", " "),
    (r"[»«><]+", " "),
    (r"(?i)\bquote from\b.*?:", " "),
    (r"(?i)\bquote:\b", " "),
    (r"(?i)&nbsp;|<br\s*/?>|</?p>", " "),
    (r"(?i)\b0\s+login to like this post\b", " "),
]
def clean_forum_noise(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text
    for pat, rep in NOISE_PATTERNS:
        t = re.sub(pat, rep, t)
    return normalize_ws(t)

assert "post_text" in df.columns, f"`post_text` column not found. Columns: {list(df.columns)}"
df["post_text"] = df["post_text"].astype(str).apply(clean_forum_noise)

def extract_forum_date_from_text(text: str):
    if not isinstance(text, str): return pd.NaT
    m = re.search(r"[A-Z][a-z]{2},\s*\d{1,2}\s+[A-Z][a-z]{2}\s+20\d{2}\s+\d{1,2}:\d{2}\s*(?:am|pm)", text)
    if not m: return pd.NaT
    try:
        return pd.to_datetime(m.group(0), errors="coerce")
    except Exception:
        return pd.NaT

if "date" not in df.columns or df["date"].notna().sum() == 0:
    df["date"] = df["post_text"].apply(extract_forum_date_from_text)
    print(f"[INFO] Embedded dates parsed from text: {df['date'].notna().sum()} / {len(df)}")

print("\n--- Sample cleaned texts ---")
for i in range(min(3, len(df))):
    print(f"{i+1}.", df['post_text'].iloc[i][:200], "\n")

# ---------- [5] MAIN PREPROCESS ----------
HAS_DATE = ("date" in df.columns) and df["date"].notna().any()
print(f"[INFO] HAS_DATE={HAS_DATE}")

if YEAR_FILTER and HAS_DATE:
    before = len(df)
    df = df[df["date"].dt.year.between(YEAR_MIN, YEAR_MAX, inclusive="both")].copy()
    print(f"[INFO] Year-filtered {YEAR_MIN}-{YEAR_MAX}: kept {len(df)}/{before}")
else:
    print("[INFO] Skipping year filter (no dates or YEAR_FILTER=False).")

df["raw_text"]   = df["post_text"].astype(str)
df["clean_text"] = df["raw_text"].apply(basic_clean)
df = df[df["clean_text"].str.len() > 20].copy()

key_parts = [df["clean_text"].astype(str)]
if "author" in df.columns: key_parts.insert(0, df["author"].astype(str))
if "title"  in df.columns: key_parts.insert(0, df["title"].astype(str))
if HAS_DATE:               key_parts.append(df["date"].dt.date.astype(str))
df["_k"] = key_parts[0]
for col in key_parts[1:]:
    df["_k"] = df["_k"] + "||" + col
before = len(df)
df = df.drop_duplicates(subset=["_k"]).drop(columns=["_k"])
after = len(df)
print(f"[INFO] Dedupe removed {before - after} → kept {after}")

# ---------- [6] SINGLISH ENRICHMENT ----------
sing_words, sing_meta = build_singdict_from_lexicon(SINGLEX_CSV)
found_terms, norm_texts, meanings = [], [], []
for t in df["raw_text"].astype(str):
    f, n = find_singlish_terms(t, sing_words)
    found_terms.append(f); norm_texts.append(n)
    m = [sing_meta[w]["description"] for w in f if w in sing_meta and sing_meta[w].get("description")]
    meanings.append(list(dict.fromkeys(m)))
df["singlish_terms"]    = found_terms
df["has_singlish"]      = df["singlish_terms"].apply(bool)
df["text_sing_norm"]    = norm_texts
df["singlish_meanings"] = meanings

# ---------- [7] PROPERTY PREPROCESS ----------
# Regex flags
if REGEX_JSONL and Path(REGEX_JSONL).exists():
    comp = compile_regexes_from_jsonl(Path(REGEX_JSONL))
    hits = [detect_regex_hits(t, comp) for t in df["clean_text"].astype(str)]
    rx_df = pd.json_normalize(hits)
    if not rx_df.empty:
        rx_df.columns = [f"rx_{c}" for c in rx_df.columns]
        df = pd.concat([df.reset_index(drop=True), rx_df.reset_index(drop=True)], axis=1)
        print(f"[INFO] Added regex flags: {len(rx_df.columns)} cols")
    else:
        print("[INFO] No regex hits normalized.")
else:
    print("[INFO] REGEX patterns not found; skipping regex flags.")

# EntityRuler spans
if ENTITYRULER and Path(ENTITYRULER).exists():
    df = add_entity_ruler_spacy(df, "clean_text", ENTITYRULER)
else:
    if "entities" not in df.columns:
        df["entities"] = [[] for _ in range(len(df))]
    print("[INFO] ENTITYRULER missing; entities set to empty lists.")

# Build wide features from entities
from collections import Counter
def _labels_and_texts(ents_list):
    labels, texts, pairs = [], [], []
    for e in (ents_list or []):
        lab = str(e.get("label","")).upper()
        txt = str(e.get("text","")).strip()
        if lab and txt:
            labels.append(lab); texts.append(txt); pairs.append((txt, lab))
    return labels, texts, pairs

all_labels=set()
labels_series, texts_series, pairs_series = [], [], []
for ents in df["entities"]:
    labs, txts, prs = _labels_and_texts(ents)
    labels_series.append(labs); texts_series.append(txts); pairs_series.append(prs)
    all_labels.update(labs)
df["prop_entity_labels"] = labels_series
df["prop_entity_texts"]  = texts_series
df["prop_entity_pairs"]  = pairs_series

all_labels = sorted(list(all_labels))
for lab in all_labels:
    df[f"has_{lab}"]    = df["prop_entity_labels"].apply(lambda L: lab in L)
    df[f"entcnt_{lab}"] = df["prop_entity_labels"].apply(lambda L: L.count(lab))
df["prop_topic_tags"] = df["prop_entity_labels"].apply(lambda L: [k for k,_ in Counter(L).most_common(5)])

# convenience policy flags on text
POLICY_TERMS = {
    "ABSD": re.compile(r"\babsd\b", re.I),
    "BSD":  re.compile(r"\bbsd\b", re.I),
    "SSD":  re.compile(r"\bssd\b", re.I),
    "LTV":  re.compile(r"\bltv\b", re.I),
    "MSR":  re.compile(r"\bmsr\b", re.I),
    "TDSR": re.compile(r"\btdsr\b", re.I),
    "HLE":  re.compile(r"\bhle\b", re.I),
}
ctext = df["clean_text"].astype(str)
for name, patt in POLICY_TERMS.items():
    df[f"flag_{name}"] = ctext.apply(lambda t, p=patt: bool(p.search(t or "")))

# ---------- [8] SAVE CORE DATASET ----------
Path(OUTDIR).mkdir(parents=True, exist_ok=True)
out_csv  = f"{OUTDIR}/forum_enriched.csv"
out_parq = f"{OUTDIR}/forum_enriched.parquet"
df.to_csv(out_csv, index=False)
try:
    df.to_parquet(out_parq, index=False)
except Exception as e:
    print(f"[WARN] Parquet write failed: {e}")

# Property-only dumps
prop_csv  = f"{OUTDIR}/forum_property_features.csv"
prop_parq = f"{OUTDIR}/forum_property_features.parquet"
df.to_csv(prop_csv, index=False)
try:
    df.to_parquet(prop_parq, index=False)
except Exception as e:
    print(f"[WARN] Parquet write failed: {e}")

# Stats JSON
stats = {
    "rows_after_clean_dedup": int(len(df)),
    "has_singlish_true": int(df["has_singlish"].sum()),
    "has_date": bool(HAS_DATE),
    "non_null_dates": int(df["date"].notna().sum()) if "date" in df.columns else 0,
    "cols": list(df.columns),
    "entity_labels_seen": all_labels,
    "rx_columns": [c for c in df.columns if c.startswith("rx_")],
}
with open(f"{OUTDIR}/preprocess_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print("Saved:", out_csv)
print("Saved:", out_parq)
print("Saved:", prop_csv)
print("Saved:", prop_parq)
print("Saved:", f"{OUTDIR}/preprocess_stats.json")



[INFO] Loaded existing patterns: 2256
[INFO] Built vocab patterns: 1025
[DONE] Wrote merged patterns → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl (skipped dupes: 795)
ENTITYRULER now: /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
[INFO] Loaded: /content/drive/MyDrive/PropInsight/raw/forums/singapore_property_forum_posts_sgexpats.csv | rows: 17362
Cols: ['thread_url', 'post_text']
[INFO] Embedded dates parsed from text: 1074 / 17362

--- Sample cleaned texts ---
1. Australian International School Singapore AIS has a long history of delivering a holistic, rounded education. What makes us truly unique is that we offer a southern hemisphere academic calendar throug 

2. Stamford American International School Stamford American International Singapore offers an holistic education for students from 2 months to 18 years of age, being the first and only school in Singapor 

3. Re: Stamfo

In [ ]:
#!python -m spacy download en_core_web_sm -q

In [ ]:
# ---------- [9] OPTIONAL NLP ENRICHMENT ----------
if RUN_NLP_ENRICHMENT:

    import spacy, os
    nlp = spacy.load("en_core_web_sm", exclude=[])
    try:
        if ENTITYRULER and Path(ENTITYRULER).exists():
            ruler = nlp.add_pipe("entity_ruler", before="ner")
            ruler.from_disk(ENTITYRULER)
    except Exception as e:
        print("[WARN] EntityRuler reattach skipped:", e)

    docs = list(nlp.pipe(df["clean_text"].astype(str).tolist(), batch_size=64, n_process=2))

    df["tokens"] = [[t.text  for t in d] for d in docs]
    df["lemmas"] = [[t.lemma_ for t in d] for d in docs]
    df["pos"]    = [[t.pos_   for t in d] for d in docs]
    df["deps"]   = [[t.dep_   for t in d] for d in docs]
    df["entities_ner"] = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
    df["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]

    df.to_parquet(f"{OUTDIR}/forum_enriched+nlp.parquet", index=False)
    print("Saved:", f"{OUTDIR}/forum_enriched+nlp.parquet")

    sent_rows=[]
    use_date = ("date" in df.columns) and df["date"].notna().any()
    for i, d in enumerate(docs):
        for j, s in enumerate(d.sents):
            row = {"post_id": i, "sent_id": j, "text": s.text,
                   "tokens": [t.text for t in s],
                   "lemmas": [t.lemma_ for t in s],
                   "pos":    [t.pos_   for t in s],
                   "deps":   [t.dep_   for t in s]}
            if use_date: row["date"] = df.iloc[i]["date"]
            sent_rows.append(row)
    pd.DataFrame(sent_rows).to_parquet(f"{OUTDIR}/forum_sentences.parquet", index=False)
    print("Saved:", f"{OUTDIR}/forum_sentences.parquet")

    if use_date:
        monthly = (
            df.assign(month=df["date"].dt.to_period("M").astype(str))
              .groupby("month", dropna=True)
              .agg(posts=("clean_text", "size"),
                   with_singlish=("has_singlish", "sum"))
              .reset_index()
        )
        monthly.to_csv(f"{OUTDIR}/monthly_counts.csv", index=False)
        print("Saved:", f"{OUTDIR}/monthly_counts.csv")
    else:
        print("[INFO] Skipping monthly trend: no usable dates.")
else:
    print("[INFO] RUN_NLP_ENRICHMENT=False → skipping NLP step.")



Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed/forum_enriched+nlp.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed/forum_sentences.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed/monthly_counts.csv


In [ ]:
# ---------- [10] VERIFICATION ----------
print("\n=== VERIFICATION ===")
print("OUTDIR:", OUTDIR)
print("Files:", sorted(os.listdir(OUTDIR))[:20])

enriched_csv = f"{OUTDIR}/forum_enriched.csv"
if os.path.exists(enriched_csv):
    df_check = pd.read_csv(enriched_csv, nrows=3)
    print("\n✅ Loaded:", enriched_csv)
    print("Columns:", df_check.columns.tolist())
else:
    print("\n❌ forum_enriched.csv not found!")

print("Has sentence parquet?", os.path.exists(f"{OUTDIR}/forum_sentences.parquet"))
print("Has enriched+nlp parquet?", os.path.exists(f"{OUTDIR}/forum_enriched+nlp.parquet"))
print("Has monthly_counts.csv?", os.path.exists(f"{OUTDIR}/monthly_counts.csv"))

stats_path = f"{OUTDIR}/preprocess_stats.json"
print("Has stats JSON?", os.path.exists(stats_path))
if os.path.exists(stats_path):
    print(json.load(open(stats_path)))



=== VERIFICATION ===
OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed
Files: ['forum_enriched+nlp.parquet', 'forum_enriched.csv', 'forum_enriched.parquet', 'forum_property_features.csv', 'forum_property_features.parquet', 'forum_sentences.parquet', 'monthly_counts.csv', 'preprocess_stats.json']

✅ Loaded: /content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed/forum_enriched.csv
Columns: ['thread_url', 'post_text', 'date', 'raw_text', 'clean_text', 'singlish_terms', 'has_singlish', 'text_sing_norm', 'singlish_meanings', 'entities', 'prop_entity_labels', 'prop_entity_texts', 'prop_entity_pairs', 'has_AGENCY&AUTHORITY', 'entcnt_AGENCY&AUTHORITY', 'has_AGENCY_AUTHORITY', 'entcnt_AGENCY_AUTHORITY', 'has_FINANCE&RATES', 'entcnt_FINANCE&RATES', 'has_HDB', 'entcnt_HDB', 'has_LEGAL&DOCS', 'entcnt_LEGAL&DOCS', 'has_LEGAL_DOCS', 'entcnt_LEGAL_DOCS', 'has_MAINTENANCE', 'entcnt_MAINTENANCE',